# Silver incremental processing
Standardize Bronze records, enforce quality rules and report newly committed Silver rows.

## Imports and project setup
Resolve the project path and load dependencies before creating widgets.

In [ ]:
import sys
from pathlib import Path

project_root = Path(dbutils.widgets.getAll().get(
    "project_root", "/Workspace/Users/allanjs3@yahoo.com.br/beverage_sales"
))
if not project_root.is_absolute() or not (project_root / "configs/silver").is_dir():
    raise ValueError("project_root must be an absolute project directory containing configs/silver.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Standard IPython support for interactive edits to driver-side modules.
get_ipython().run_line_magic("reload_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

from utils.execution import finish_execution, start_execution, to_json
from utils.presentation import render_execution_summary
from utils.widgets import build_table_dropdown, resolve_table_selection
from transformations.silver.configuration import load_silver_config
from transformations.silver.processing import process_sources, inspect_new_rows

## Widgets
Prepare project and table parameters. Recreate the table selector in the following cell.

In [ ]:
dbutils.widgets.text("project_root", "/Workspace/Users/allanjs3@yahoo.com.br/beverage_sales", "Project Root")
previous_widgets = dbutils.widgets.getAll()
table_dropdown = build_table_dropdown(
    project_root / "configs/silver",
    previous_value=previous_widgets.get("table_name"),
    reset=globals().get("_selector_project_root", str(project_root)) != str(project_root),
)
selector_signature = ("multiselect", str(project_root), tuple(table_dropdown["choices"]))
selector_changed = (
    globals().get("_selector_project_root", str(project_root)) != str(project_root)
    or globals().get("_selector_signature", selector_signature) != selector_signature
)
_selector_project_root = str(project_root)
_selector_signature = selector_signature
# Widget removal and creation must occur in separate Databricks cells.
if selector_changed and "table_name" in previous_widgets:
    dbutils.widgets.remove("table_name")
    table_dropdown["default_value"] = "Select a table"

In [ ]:
if str(Path(dbutils.widgets.get("project_root"))) != _selector_project_root:
    raise ValueError("Refresh the parameter cells after changing project_root.")
dbutils.widgets.multiselect(
    "table_name", table_dropdown["default_value"].split(",")[0], table_dropdown["choices"], "Silver Tables"
)

## Table configuration
Load selected tables and the approved channel reference used by sales.

In [ ]:
if str(Path(dbutils.widgets.get("project_root"))) != _selector_project_root:
    raise ValueError("Refresh the parameter cells after changing Project Root.")
table_name = resolve_table_selection(
    dbutils.widgets.get("table_name"), table_dropdown["choices"], label="Silver Tables"
)
config = load_silver_config(project_root / "configs/silver", table_name)
print("Selected tables: " + ", ".join(source["target"].split(".")[-1] for source in config["sources"]))
execution = start_execution()

## Incremental processing
Validate and write each pending batch with one stream per table, then synchronize changed descriptions.

In [ ]:
result = process_sources(spark, dbutils, config, execution)

## Quality observations and outcome
Summarize newly written records and non-blocking quality warnings.

In [ ]:
if result["has_new_data"]:
    result = inspect_new_rows(spark, config, result)
result = finish_execution(result)
displayHTML(render_execution_summary(result, "Silver"))

## Automation result
Return the structured result to the calling job or notebook.

In [ ]:
dbutils.jobs.taskValues.set(key="has_new_data", value=bool(result["has_new_data"]))
dbutils.notebook.exit(to_json(result))
